# 01 — Data Quality Analysis
**NovaCred Credit Application Governance Audit**  


This notebook systematically audits the raw credit application dataset for:
- **Completeness** — missing / null values across all fields
- **Consistency** — inconsistent encodings, formatting, and data types
- **Validity** — values that violate business or logical rules
- **Accuracy** — suspicious or impossible values

All issues are quantified with record counts and percentages, and remediation is demonstrated in code.

## 0. Setup & Imports

In [57]:
import json
import re
import hashlib
import warnings
from collections import Counter, defaultdict
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 130

# ── reproducibility
SEED = 42
np.random.seed(SEED)

print('Libraries loaded ✓')

Libraries loaded ✓


## 1. Load Raw Data

In this step, we load the raw nested JSON dataset from the project's data directory to begin our quality audit. By reading the file into Python, we can safely confirm the initial dataset contains exactly 502 credit applications. We also inspect the top-level keys to understand the complex nested structure before we flatten it into a readable table for our column-level validation.

In [58]:
# Tell Python to look in the data folder
RAW_PATH = '../data/raw_credit_applications.json'

with open(RAW_PATH, 'r') as f:
    raw = json.load(f)

print(f'Raw records loaded : {len(raw):,}')
print(f'Top-level keys     : {list(raw[0].keys())}')

Raw records loaded : 502
Top-level keys     : ['_id', 'applicant_info', 'financials', 'spending_behavior', 'decision', 'processing_timestamp']


### 1.1 Flatten to Tabular Format

In this phase, we evaluate the uniqueness and completeness of our dataset. First, we identify any records sharing the same application ID. Then, we scan for missing essential fields like emails, SSNs, or incomes. Rather than immediately deleting these broken records, we will append a quarantine flag to them. This ensures our Data Scientist can easily filter them out during model training while the Governance Officer maintains a proper audit trail of the missing data.

In [59]:
def flatten_record(rec):
    """Flatten one nested JSON record into a single dict."""
    ai   = rec.get('applicant_info', {}) or {}
    fin  = rec.get('financials', {})      or {}
    dec  = rec.get('decision', {})        or {}
    sb   = rec.get('spending_behavior', []) or []

    # aggregate spending categories
    spend_cats = [s.get('category') for s in sb if s.get('category')]
    spend_total = sum(s.get('amount', 0) or 0 for s in sb)

    return {
        '_id':                   rec.get('_id'),
        'full_name':             ai.get('full_name'),
        'email':                 ai.get('email'),
        'ssn':                   ai.get('ssn'),
        'ip_address':            ai.get('ip_address'),
        'gender_raw':            ai.get('gender'),
        'date_of_birth_raw':     ai.get('date_of_birth'),
        'zip_code':              ai.get('zip_code'),
        'annual_income_raw':     fin.get('annual_income'),
        'credit_history_months': fin.get('credit_history_months'),
        'debt_to_income':        fin.get('debt_to_income'),
        'savings_balance':       fin.get('savings_balance'),
        'loan_approved':         dec.get('loan_approved'),
        'interest_rate':         dec.get('interest_rate'),
        'approved_amount':       dec.get('approved_amount'),
        'rejection_reason':      dec.get('rejection_reason'),
        'spend_categories':      spend_cats,
        'spend_total_monthly':   spend_total,
        'loan_purpose':          rec.get('loan_purpose'),
        'processing_timestamp':  rec.get('processing_timestamp'),
    }

df_raw = pd.DataFrame([flatten_record(r) for r in raw])
print(f'Shape: {df_raw.shape}')
df_raw.head(3)

Shape: (502, 20)


,_id,full_name,email,ssn,ip_address,gender_raw,date_of_birth_raw,zip_code,annual_income_raw,credit_history_months,debt_to_income,savings_balance,loan_approved,interest_rate,approved_amount,rejection_reason,spend_categories,spend_total_monthly,loan_purpose,processing_timestamp
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036,73000,23,0.20,31212,False,NaN,NaN,algorithm_risk_score,"[Shopping, Rent, Alcohol]",1517,None,2024-01-15T00:00:00Z
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,M,1992-03-31,10032,78000,51,0.18,17915,False,NaN,NaN,algorithm_risk_score,"[Rent, Dining, Healthcare]",947,None,None
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075,61000,41,0.21,37909,True,3.7,59000.0,None,[Rent],109,vacation,None


---
## 2. Issue #1 — Duplicate Records
**Dimension: Consistency / Uniqueness**

In this phase, we evaluate the dataset for consistency by checking for duplicate application IDs. We identify 4 records involved in duplicate ID groups (app_042 and app_001). After inspecting these duplicates side-by-side to understand the data entry errors, we apply our first remediation step by keeping only the first occurrence of each application, resulting in a clean base of 500 unique records.

In [60]:
# ── duplicate application IDs
id_counts = df_raw['_id'].value_counts()
dup_ids = id_counts[id_counts > 1]

print(f'Total records               : {len(df_raw):,}')
print(f'Unique application IDs      : {df_raw["_id"].nunique():,}')
print(f'Duplicate ID groups         : {len(dup_ids)}')
print(f'Records involved in dups    : {dup_ids.sum():,}  '
      f'({dup_ids.sum()/len(df_raw)*100:.1f}%)')
print()
print('Duplicated IDs:')
print(dup_ids)

# also check full-row duplicates
full_dups = df_raw.duplicated(subset=['full_name','annual_income_raw','debt_to_income','zip_code'], keep=False)
print(f'\nRecords sharing name+income+DTI+ZIP: {full_dups.sum()} '
      f'({full_dups.sum()/len(df_raw)*100:.1f}%)')

Total records               : 502
Unique application IDs      : 500
Duplicate ID groups         : 2
Records involved in dups    : 4  (0.8%)

Duplicated IDs:
_id
app_042    2
app_001    2
Name: count, dtype: int64

Records sharing name+income+DTI+ZIP: 2 (0.4%)


In [61]:
# Inspect the duplicated records side by side
for dup_id in dup_ids.index:
    print(f'\n=== {dup_id} ===')
    display(df_raw[df_raw['_id'] == dup_id][['_id','full_name','annual_income_raw',
                                              'zip_code','loan_approved','rejection_reason']])


=== app_042 ===


,_id,full_name,annual_income_raw,zip_code,loan_approved,rejection_reason
8,app_042,Joseph Lopez,69000,10044,False,algorithm_risk_score
354,app_042,Joseph Lopez,69000,10044,False,algorithm_risk_score



=== app_001 ===


,_id,full_name,annual_income_raw,zip_code,loan_approved,rejection_reason
383,app_001,Stephanie Nguyen,102000,90230,False,high_dti_ratio
455,app_001,Stephanie Nguyen,102000,None,False,high_dti_ratio


In [62]:
# ── REMEDIATION: keep first occurrence of each ID
df = df_raw.drop_duplicates(subset='_id', keep='first').copy()
print(f'Records after deduplication: {len(df):,}  '
      f'(removed {len(df_raw) - len(df)} duplicates)')

Records after deduplication: 500  (removed 2 duplicates)


---
## 3. Issue #2 — Missing / Incomplete Fields
**Dimension: Completeness**

Next, we examine the dataset for completeness by calculating the total missing values across all columns. We specifically focus on a subset of critical fields—email, SSN, gender, date of birth, and annual income—and identify 12 specific records that are missing at least one of these essential data points. Instead of immediately deleting these incomplete applications, we apply a quarantine pattern by adding a specific flag to them, ensuring they can be tracked for governance audit trails while being safely filtered out by the Data Scientist during model training.

In [63]:
# Replace empty strings with NaN so isnull() catches them
df.replace('', np.nan, inplace=True)

null_counts = df.isnull().sum().sort_values(ascending=False)
null_pct    = (null_counts / len(df) * 100).round(2)

missing_summary = pd.DataFrame({
    'Missing Count': null_counts,
    'Missing %': null_pct
})
print(missing_summary[missing_summary['Missing Count'] > 0].to_string())

                      Missing Count  Missing %
loan_purpose                    450       90.0
processing_timestamp            438       87.6
rejection_reason                292       58.4
approved_amount                 208       41.6
interest_rate                   208       41.6
email                             7        1.4
annual_income_raw                 5        1.0
ssn                               4        0.8
ip_address                        4        0.8
date_of_birth_raw                 4        0.8
gender_raw                        2        0.4
zip_code                          1        0.2


In [64]:
# Classify records with >=1 missing critical field
critical_fields = ['email','ssn','gender_raw','date_of_birth_raw','annual_income_raw']
incomplete_mask = df[critical_fields].isnull().any(axis=1)
print(f'Records missing at least one critical field: '
      f'{incomplete_mask.sum()} ({incomplete_mask.sum()/len(df)*100:.1f}%)')

Records missing at least one critical field: 12 (2.4%)


In [65]:
# ── REMEDIATION DEMO: flag incomplete records (quarantine pattern)
df['dq_flag_missing'] = incomplete_mask
print('Quarantine flag added: dq_flag_missing')
print(df['dq_flag_missing'].value_counts())

Quarantine flag added: dq_flag_missing
dq_flag_missing
False    488
True      12
Name: count, dtype: int64


---
## 4. Issue #3 — Inconsistent Gender Coding
**Dimension: Consistency / Standardisation**

In this phase, we address data consistency by standardizing categorical encodings, specifically focusing on the gender field. Our initial scan reveals inconsistent data entry, with 111 records using abbreviated codes like 'M' and 'F' alongside full words like 'Male' and 'Female'. To resolve this, we map all raw values to a canonical standard of strictly 'Male' and 'Female', producing a clean distribution of 247 Males and 251 Females. This remediation is a critical prerequisite for our Data Scientist, as messy categorical strings would break the calculation of the Disparate Impact Ratio during the subsequent bias analysis phase.

In [66]:
gender_counts = df['gender_raw'].value_counts(dropna=False)
print('Raw gender value distribution:')
print(gender_counts.to_string())



Raw gender value distribution:
gender_raw
Male      194
Female    193
F          58
M          53
NaN         2


In [67]:
# ── REMEDIATION: map to canonical values
GENDER_MAP = {
    'Male'  : 'Male',
    'M'     : 'Male',
    'Female': 'Female',
    'F'     : 'Female',
}
df['gender'] = df['gender_raw'].map(GENDER_MAP)  # unmapped → NaN

print('Standardised gender distribution:')
print(df['gender'].value_counts(dropna=False))

inconsistent_n = df['gender_raw'].isin(['M','F']).sum()
null_n = df['gender'].isna().sum()
print(f'\nAbbreviated codes normalised : {inconsistent_n}')
print(f'Remaining null after mapping : {null_n}')

Standardised gender distribution:
gender
Female    251
Male      247
NaN         2
Name: count, dtype: int64

Abbreviated codes normalised : 111
Remaining null after mapping : 2


---
## 5. Issue #4 — Inconsistent Date Formats
**Dimension: Consistency / Validity**

Continuing with our consistency checks, we analyze the date_of_birth field and discover a messy mix of four different date formats, including European (DD/MM/YYYY) and American (MM/DD/YYYY) styles. To resolve this validity issue, we apply a custom parsing function to force all entries into a single standardized datetime format. We successfully parsed 99.2% of the records (leaving only 4 unparseable missing values) and used this clean data to calculate a derived age column, confirming all applicants fall within a logical age range of 22 to 65 years old.

In [68]:
DATE_FORMATS = [
    ('%Y-%m-%d', 'ISO-8601 (YYYY-MM-DD)'),
    ('%d/%m/%Y', 'DD/MM/YYYY'),
    ('%m/%d/%Y', 'MM/DD/YYYY'),
    ('%Y/%m/%d', 'YYYY/MM/DD'),
]

def detect_date_format(val):
    if pd.isna(val):
        return 'NULL'
    for fmt, label in DATE_FORMATS:
        try:
            datetime.strptime(str(val), fmt)
            return label
        except ValueError:
            continue
    return 'Unknown'

df['dob_format_detected'] = df['date_of_birth_raw'].apply(detect_date_format)
fmt_counts = df['dob_format_detected'].value_counts()
print('Date-of-birth format distribution:')
print(fmt_counts.to_string())



Date-of-birth format distribution:
dob_format_detected
ISO-8601 (YYYY-MM-DD)    339
DD/MM/YYYY                75
YYYY/MM/DD                56
MM/DD/YYYY                26
NULL                       4


In [69]:
# ── REMEDIATION: parse to a single canonical datetime
def parse_dob(val):
    if pd.isna(val):
        return pd.NaT
    for fmt, _ in DATE_FORMATS:
        try:
            return datetime.strptime(str(val), fmt)
        except ValueError:
            continue
    return pd.NaT

df['date_of_birth'] = df['date_of_birth_raw'].apply(parse_dob)
REFERENCE_DATE = datetime(2024, 1, 1)
df['age'] = ((REFERENCE_DATE - df['date_of_birth']).dt.days / 365.25).round(1)

successfully_parsed = df['date_of_birth'].notna().sum()
print(f'Successfully parsed : {successfully_parsed}/{len(df)} '
      f'({successfully_parsed/len(df)*100:.1f}%)')
print(f'Still unparseable   : {df["date_of_birth"].isna().sum()}')
print(f'Age range: {df["age"].min():.0f} – {df["age"].max():.0f}')

Successfully parsed : 496/500 (99.2%)
Still unparseable   : 4
Age range: 22 – 65


---
## 6. Issue #5 — Inconsistent Data Types: `annual_income`
**Dimension: Consistency / Validity**

In the next phase of our data validity process, we check for inconsistent data types within critical financial fields, which is a specific requirement for our governance audit. We discover that 8 records have their annual income incorrectly stored as text strings rather than numbers. To ensure accuracy in downstream financial calculations and risk modeling, we remediate this by coercing these string values into proper numeric formats. The transformation is successful, allowing us to safely generate statistical summaries that confirm the income values now logically compute and fall within a realistic range.

In [70]:
# Detect Python type of each raw income value
df['income_dtype'] = df['annual_income_raw'].apply(
    lambda x: type(x).__name__ if x is not None else 'NoneType'
)
dtype_counts = df['income_dtype'].value_counts()
print('annual_income raw type distribution:')
print(dtype_counts.to_string())

# Show offending records
str_income = df[df['income_dtype'] == 'str'][['_id','annual_income_raw']]
print(f'\nString-typed income records ({len(str_income)}):')
print(str_income.to_string())

none_income = df[df['income_dtype'] == 'NoneType'][['_id','annual_income_raw']]
print(f'\nNull income records ({len(none_income)}):')
print(none_income.to_string())

annual_income raw type distribution:
income_dtype
int         486
str           8
NoneType      5
float         1

String-typed income records (8):
         _id annual_income_raw
92   app_088             55000
146  app_135             65000
171  app_446             73000
306  app_389             51000
334  app_026             72000
431  app_312             80000
447  app_180            111000
490  app_224             93000

Null income records (5):
         _id annual_income_raw
76   app_436              None
94   app_421              None
99   app_479              None
141  app_463              None
149  app_449              None


In [71]:
# ── REMEDIATION: coerce to numeric, leave uncoerceable as NaN
df['annual_income'] = pd.to_numeric(df['annual_income_raw'], errors='coerce')

coerced   = len(str_income)
remaining_null = df['annual_income'].isna().sum()
print(f'String values coerced to numeric : {coerced}')
print(f'Still NaN after coercion         : {remaining_null} (were already null)')
print(f'Income stats after coercion:')
print(df['annual_income'].describe().round(0))

String values coerced to numeric : 8
Still NaN after coercion         : 5 (were already null)
Income stats after coercion:
count       495.0
mean      82694.0
std       28139.0
min           0.0
25%       63000.0
50%       81000.0
75%      101000.0
max      171000.0
Name: annual_income, dtype: float64


---
## 7. Issue #6 — Invalid / Impossible Values
**Dimension: Validity / Accuracy**

In the final stage of our validity checks, we scan the dataset for logically impossible values that would skew our financial risk models. Our audit successfully caught 2 records with a negative credit history and 1 record with a negative savings balance. Additionally, we identified 1 application with an impossible Debt-to-Income (DTI) ratio greater than 1.0, and 4 records containing invalid email address formats. To safely remediate these errors while preserving the data governance audit trail, we generate specific boolean flags for these rows and nullify the impossible numerical values by converting them to NaN. This ensures they do not corrupt the Data Scientist's downstream fairness metrics.

In [72]:
# ── 7a. Negative credit history months
neg_credit = df[df['credit_history_months'] < 0][['_id','credit_history_months','loan_approved']]
print(f'Records with negative credit_history_months: {len(neg_credit)}')
print(neg_credit.to_string())

Records with negative credit_history_months: 2
         _id  credit_history_months  loan_approved
137  app_043                    -10           True
162  app_156                     -3          False


In [73]:
# ── 7a-bis. Negative savings balance
neg_savings = df[pd.to_numeric(df['savings_balance'], errors='coerce') < 0][['_id','savings_balance','loan_approved']]
print(f'Records with negative savings_balance: {len(neg_savings)}')
if len(neg_savings):
    print(neg_savings.to_string())

Records with negative savings_balance: 1
         _id  savings_balance  loan_approved
159  app_290            -5000           True


In [74]:
# ── 7b. Unreasonably extreme income values (< minimum wage annual or > 5 million)
INCOME_MIN = 0   # below min-wage threshold
INCOME_MAX = 1_000_000
extreme_income = df[
    df['annual_income'].notna() &
    ((df['annual_income'] < INCOME_MIN) | (df['annual_income'] > INCOME_MAX))
][['_id','annual_income','loan_approved']]
print(f'Records with extreme income values: {len(extreme_income)}')
if len(extreme_income):
    print(extreme_income.to_string())

Records with extreme income values: 0


In [75]:
# ── 7c. Debt-to-income > 1.0 (impossible: more debt than income)
impossible_dti = df[df['debt_to_income'] > 1.0][['_id','debt_to_income']]
print(f'Records with DTI > 1.0: {len(impossible_dti)}')
if len(impossible_dti):
    print(impossible_dti)

# ── 7d. Approved loans missing interest rate or approved amount
approved_df = df[df['loan_approved'] == True]
missing_rate   = approved_df['interest_rate'].isna().sum()
missing_amount = approved_df['approved_amount'].isna().sum()
print(f'\nApproved loans missing interest_rate   : {missing_rate}')
print(f'Approved loans missing approved_amount : {missing_amount}')

# ── 7e. Rejected loans that (oddly) have an approved amount
rejected_with_amount = df[
    (df['loan_approved'] == False) &
    (df['approved_amount'].notna())
][['_id','loan_approved','approved_amount','rejection_reason']]
print(f'Rejected loans with approved_amount set: {len(rejected_with_amount)}')
if len(rejected_with_amount):
    print(rejected_with_amount)

Records with DTI > 1.0: 1
         _id  debt_to_income
316  app_402            1.85

Approved loans missing interest_rate   : 0
Approved loans missing approved_amount : 0
Rejected loans with approved_amount set: 0


In [76]:
# ── 7f. Applicants whose computed age is implausible (< 18 or > 100)
underage  = df[df['age'] < 18][['_id','age','date_of_birth','loan_approved']]
too_old   = df[df['age'] > 100][['_id','age','date_of_birth','loan_approved']]
print(f'Applicants under 18 : {len(underage)}')
if len(underage):
    print(underage)
print(f'Applicants over 100 : {len(too_old)}')
if len(too_old):
    print(too_old)

Applicants under 18 : 0
Applicants over 100 : 0


In [77]:
# ── 7g. Invalid email format check
EMAIL_REGEX = re.compile(r'^[^@\s]+@[^@\s]+\.[^@\s]+$')
df['email_valid'] = df['email'].apply(
    lambda x: bool(EMAIL_REGEX.match(str(x))) if pd.notna(x) else False
)
invalid_emails = df[~df['email_valid'] & df['email'].notna()]
print(f'Records with invalid email format: {len(invalid_emails)}')
if len(invalid_emails):
    print(invalid_emails[['_id','email']].head(10))

Records with invalid email format: 4
         _id                   email
138  app_204  mike johnson@gmail.com
181  app_299   test.user.outlook.com
276  app_068        john.doe@invalid
369  app_146            sarah.smith@


In [78]:
# ── REMEDIATION: flag all validity issues
df['dq_flag_neg_credit']   = df['credit_history_months'] < 0
df['dq_flag_extreme_income'] = (
    df['annual_income'].notna() &
    ((df['annual_income'] < INCOME_MIN) | (df['annual_income'] > INCOME_MAX))
)
df['dq_flag_impossible_dti'] = df['debt_to_income'] > 1.0
df['dq_flag_underage']       = df['age'] < 18

# Replace negative credit history with NaN
df.loc[df['credit_history_months'] < 0, 'credit_history_months'] = np.nan


df['dq_flag_neg_savings'] = pd.to_numeric(df['savings_balance'], errors='coerce') < 0
df.loc[df['dq_flag_neg_savings'], 'savings_balance'] = np.nan


print('Validity flags added and negative credit values nullified.')



Validity flags added and negative credit values nullified.


---
## 8. Comprehensive Data Quality Summary

In this section, we consolidate all our findings into a final, comprehensive data quality summary table. This table explicitly maps every identified issue,such as: duplicate records, inconsistent date formats, string-typed incomes, and missing fields to its corresponding Data Quality Dimension: Uniqueness, Consistency, Completeness, and Validity. By quantifying the exact number of affected records and calculating the percentage of the dataset impacted for each category, we fulfill the project's strict auditing requirements and provide a clear, executive-level overview of the dataset's health before handing it off to the Data Scientist.

In [79]:
# Count all DQ flags
dq_cols = [c for c in df.columns if c.startswith('dq_flag_')]

summary_rows = []
summary_rows.append(('Duplicate Records',        'Uniqueness',   len(df_raw)-len(df),      (len(df_raw)-len(df))/len(df_raw)*100))
summary_rows.append(('Inconsistent Gender Codes', 'Consistency',  df['gender_raw'].isin(['M','F']).sum(), df['gender_raw'].isin(['M','F']).sum()/len(df)*100))
summary_rows.append(('Null / Empty Gender',       'Completeness', df['gender'].isna().sum(), df['gender'].isna().sum()/len(df)*100))
summary_rows.append(('Mixed DOB Formats (non-ISO)','Consistency', (df['dob_format_detected'] != 'ISO-8601 (YYYY-MM-DD)') .sum(), (df['dob_format_detected'] != 'ISO-8601 (YYYY-MM-DD)').sum()/len(df)*100))
summary_rows.append(('String-typed Income',       'Validity',     (df['income_dtype']=='str').sum(), (df['income_dtype']=='str').sum()/len(df)*100))
summary_rows.append(('Null Income',               'Completeness', df['annual_income'].isna().sum(), df['annual_income'].isna().sum()/len(df)*100))
summary_rows.append(('Missing Email',             'Completeness', df['email'].isna().sum(), df['email'].isna().sum()/len(df)*100))
summary_rows.append(('Negative Credit History',   'Validity',     (df['credit_history_months'] < 0).sum() if 'credit_history_months' in df.columns else df['dq_flag_neg_credit'].sum(), 0))
summary_rows.append(('Underage Applicants',       'Validity',     df['dq_flag_underage'].sum(),      df['dq_flag_underage'].sum()/len(df)*100))


dq_summary = pd.DataFrame(summary_rows,
    columns=['Issue', 'DQ Dimension', 'Affected Records', '% of Dataset'])
dq_summary['% of Dataset'] = dq_summary['% of Dataset'].round(2)
print('=== DATA QUALITY SUMMARY ===')
print(dq_summary.to_string(index=False))

=== DATA QUALITY SUMMARY ===
                      Issue DQ Dimension  Affected Records  % of Dataset
          Duplicate Records   Uniqueness                 2           0.4
  Inconsistent Gender Codes  Consistency               111          22.2
        Null / Empty Gender Completeness                 2           0.4
Mixed DOB Formats (non-ISO)  Consistency               161          32.2
        String-typed Income     Validity                 8           1.6
                Null Income Completeness                 5           1.0
              Missing Email Completeness                 7           1.4
    Negative Credit History     Validity                 0           0.0
        Underage Applicants     Validity                 0           0.0


---
## 9. Save Clean Dataset

In this final step, we prepare the dataset for handoff by dropping all intermediate helper columns (such as raw text fields and temporary format trackers) that were created during the audit process. We then export the finalized, highly-structured dataframe into a CSV file named clean_credit_applications.csv within the project's data directory. The resulting dataset contains exactly 500 validated records across 26 clean columns, providing a foundation for the Data Scientist's forthcoming bias analysis and fairness testing.

In [80]:
# Drop raw / intermediate helper columns before saving
cols_to_drop = ['gender_raw','date_of_birth_raw','annual_income_raw',
                'income_dtype','dob_format_detected','email_valid',
                'spend_categories']  # keep structured cols instead

df_clean = df.drop(columns=[c for c in cols_to_drop if c in df.columns])

CLEAN_PATH = '../data/clean_credit_applications.csv'
df_clean.to_csv(CLEAN_PATH, index=False)
print(f'Clean dataset saved to: {CLEAN_PATH}')
print(f'Shape: {df_clean.shape}')
df_clean.head(3)

Clean dataset saved to: ../data/clean_credit_applications.csv
Shape: (500, 26)


,_id,full_name,email,ssn,ip_address,zip_code,credit_history_months,debt_to_income,savings_balance,loan_approved,...,dq_flag_missing,gender,date_of_birth,age,annual_income,dq_flag_neg_credit,dq_flag_extreme_income,dq_flag_impossible_dti,dq_flag_underage,dq_flag_neg_savings
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,10036,23.0,0.20,31212.0,False,...,False,Male,2001-03-09,22.8,73000.0,False,False,False,False,False
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,10032,51.0,0.18,17915.0,False,...,False,Male,1992-03-31,31.8,78000.0,False,False,False,False,False
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,10075,41.0,0.21,37909.0,True,...,False,Male,1989-10-24,34.2,61000.0,False,False,False,False,False


## 10. Summary of Findings & Remediation

| # | Issue | Dimension | Records Affected | Remediation Applied |
|---|-------|-----------|-----------------|---------------------|
| 1 | Duplicate `_id` records | Uniqueness | 4 (2 groups) | Drop duplicate IDs, keep first |
| 2 | Missing critical fields (email, SSN, DOB, income) | Completeness | ~20 records across fields | Quarantine flag `dq_flag_missing` |
| 3 | Inconsistent gender coding (`Male`/`M`, `Female`/`F`) | Consistency | 111 records | Canonical mapping to `Male`/`Female` |
| 4 | Mixed date-of-birth formats (4 formats detected) | Consistency | 161 records | Uniform parse → ISO datetime |
| 5 | `annual_income` stored as string | Validity | 8 records | `pd.to_numeric(errors='coerce')` |
| 6 | Negative `credit_history_months` | Validity | 2 records | Set to NaN, add validity flag |
| 7 | Negative `savings_balance` | Validity | 1 record | Set to NaN, add validity flag |
| 8 | Debt-to-Income ratio > 1.0 | Validity | 1 record | Add validity flag `dq_flag_impossible_dti` |
| 9 | Invalid email formats | Validity | 4 records | Flagged via Regex pattern matching |

### Key Recommendations
1. **Schema enforcement**: Introduce a JSON Schema validator at ingestion to reject records with wrong types or empty required fields.  
2. **Canonical look-up tables**: Enforce a single gender encoding at point of collection; use controlled vocabulary dropdowns.  
3. **Date standardisation**: Mandate ISO-8601 in API contracts; parse and reject non-conforming inputs.  
4. **Data retention policy**: SSNs, IP addresses, and email addresses must have defined retention limits per GDPR Article 5(1)(e).
5. **Business Logic & Format Validation**: Implement front-end validation rules to prevent impossible financial values (e.g., negative savings balances, DTI > 1.0) and enforce strict Regex patterns for email inputs.